# Smoke Test SVLM — Skripsi Alif

**Tujuan.** Membuktikan pipeline inferensi berjalan ujung ke ujung, dan menilai kualitas
bahasa Indonesia keluaran model. **Bukan** untuk mengambil hasil penelitian.

Lima chip, dua model (Qwen2.5-VL-3B dan InternVL3-2B), dua konfigurasi penyajian masukan.
Keluaran disimpan **apa adanya**, tanpa diurai dan tanpa dirapikan.

### Sebelum mulai

1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Pastikan di Google Drive sudah ada folder `SKRIPSI ALIF` yang memuat (di mana pun
   letaknya di dalam folder itu, nama berkasnya yang dicari, bukan susunan foldernya):
   - `prompt_a_<sidik>.txt` dan `prompt_b_<sidik>.txt`  — dari `KOMPONEN_4\Output Prompt`
   - `sampel_kecil.csv`                                 — dari `KOMPONEN_3\Output Sampel`
   - folder `pra\`, `pasca\`, `gabung\` berisi chip     — dari `KOMPONEN_3\Output Chip`
3. Jalankan sel 1 sampai 9 berurutan.

### Yang dilaporkan balik

Berkas `smoke_test_mentah.jsonl` (sel 9 menyimpannya ke Drive sekaligus mengunduhnya).
Kalau ada sel yang gagal: sebutkan **nomor selnya**, salin **baris error paling bawah**,
dan jangan diperbaiki sendiri dulu.

## Sel 1 — periksa GPU

In [ ]:
!nvidia-smi

In [ ]:
import sys, torch

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())

if not torch.cuda.is_available():
    print("\n>>> GPU TIDAK AKTIF. Runtime > Change runtime type > T4 GPU > Save,")
    print(">>> lalu jalankan ulang sel ini.")
else:
    p = torch.cuda.get_device_properties(0)
    print("gpu    :", p.name, f"| {p.total_memory/1e9:.1f} GB")

## Sel 2 — pasang pustaka

Sekitar 1–3 menit. Peringatan *dependency conflict* dari pip boleh diabaikan; yang
menentukan adalah sel 5 dan sel 6 berhasil memuat model.

**Versi `transformers` dipaku di 4.51.3.** Versi yang lebih baru memakai atribut
`all_tied_weights_keys` saat memuat bobot, sedangkan kode jarak jauh InternVL3 belum
menyediakannya — itulah penyebab kegagalan `AttributeError` pada percobaan sebelumnya.
Qwen2.5-VL berjalan normal pada 4.51.3.

Kalau Colab menampilkan tombol **RESTART SESSION**, tekan, lalu lanjutkan dari sel 3
(sel 2 tidak perlu diulang).

In [ ]:
!pip install -q "transformers==4.51.3" accelerate timm einops sentencepiece
import transformers
print("transformers:", transformers.__version__)
print("kalau muncul tombol RESTART SESSION, tekan dulu, lalu lanjut ke sel 3")

## Sel 3 — sambungkan Google Drive dan periksa isinya

Sel ini hanya melihat, belum memuat apa pun. Kalau ada berkas yang belum lengkap,
sel ini yang akan memberitahu, bukan sel-sel berikutnya.

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

BASIS = Path("/content/drive/MyDrive/SKRIPSI ALIF")
N_CHIP = 5
MODEL_QWEN = "Qwen/Qwen2.5-VL-3B-Instruct"
MODEL_INTERNVL = "OpenGVLab/InternVL3-2B"
MAX_NEW_TOKENS = 512

print("folder basis :", BASIS, "| ada:", BASIS.is_dir())
if not BASIS.is_dir():
    print("\n>>> Folder tidak ditemukan. Isi MyDrive Anda:")
    for x in sorted(Path("/content/drive/MyDrive").iterdir())[:40]:
        print("   ", x.name)
    raise FileNotFoundError(BASIS)

def cari_semua(pola):
    return sorted(BASIS.rglob(pola))

temuan = {
    "prompt_a": cari_semua("prompt_a_*.txt"),
    "prompt_b": cari_semua("prompt_b_*.txt"),
    "sampel":   cari_semua("sampel_kecil.csv"),
    "chip_pra":    cari_semua("*_pra.png"),
    "chip_pasca":  cari_semua("*_pasca.png"),
    "chip_gabung": cari_semua("*_gabung.png"),
}

print("\nHasil penelusuran di dalam", BASIS.name, ":")
for k, v in temuan.items():
    tanda = "OK" if v else "!!"
    contoh = v[0].relative_to(BASIS) if v else "(tidak ada)"
    print(f"  {tanda} {k:<13}{len(v):>5}   contoh: {contoh}")

kurang = [k for k, v in temuan.items() if not v]
if kurang:
    print("\n>>> BELUM LENGKAP:", ", ".join(kurang))
    print(">>> Unggah berkas yang kurang ke dalam folder SKRIPSI ALIF, lalu ulangi sel ini.")
else:
    print("\nSemua berkas yang dibutuhkan sudah ada.")

## Sel 4 — muat prompt dan tentukan lima chip

Chip dipilih dari `sampel_kecil.csv`, jadi komposisi kelasnya mengikuti sampel evaluasi
manusia. Hanya chip yang ketiga variannya (`pra`, `pasca`, `gabung`) lengkap yang dipakai.

In [ ]:
import pandas as pd

import re

def nomor_versi(f):
    m = re.search(r"_v(\d+)_", f.name)
    return int(m.group(1)) if m else 1

def pilih(daftar):
    # utamakan nomor versi tertinggi; kalau seri, yang paling baru diubah
    return max(daftar, key=lambda f: (nomor_versi(f), f.stat().st_mtime))

print("Berkas prompt yang ada di Drive:")
for f in sorted(temuan["prompt_a"] + temuan["prompt_b"], key=lambda x: x.name):
    print(f"   v{nomor_versi(f)}  {f.name}")

f_a, f_b = pilih(temuan["prompt_a"]), pilih(temuan["prompt_b"])
PROMPT = {"A": f_a.read_text(encoding="utf-8"), "B": f_b.read_text(encoding="utf-8")}
SIDIK = f_b.stem.split("_")[-1]
VERSI = max(nomor_versi(f_a), nomor_versi(f_b))
print("\nprompt dipakai :", f_a.name, "|", f_b.name)
if VERSI < 2:
    print("\n" + "!" * 70)
    print("!! PROMPT YANG DIPAKAI MASIH v1.")
    print("!! Salin folder KOMPONEN_4\\Output Prompt yang baru ke Drive lalu ulangi sel 3 dan 4.")
    print("!" * 70)

peta = {}
for kunci in ("chip_pra", "chip_pasca", "chip_gabung"):
    for f in temuan[kunci]:
        peta[f.name] = f

sampel = pd.read_csv(temuan["sampel"][0])

def trio_lengkap(r):
    nama_gabung = str(r["chip_pra"]).replace("_pra", "_gabung")
    t = [peta.get(r["chip_pra"]), peta.get(r["chip_pasca"]), peta.get(nama_gabung)]
    return t if all(t) else None

# ambil chip secara BERGILIR antarkelas. Kalau lima chip pertama kebetulan
# hampir semuanya no-damage, model yang selalu menjawab "tidak_rusak" akan
# tampak benar padahal tidak membedakan apa pun.
antre = {k: [r for _, r in sampel[sampel["kelas_xbd"] == k].iterrows()]
         for k in sampel["kelas_xbd"].unique()}
CHIP = []
while len(CHIP) < N_CHIP and any(antre.values()):
    for k in list(antre):
        if not antre[k] or len(CHIP) >= N_CHIP:
            continue
        r = antre[k].pop(0)
        t = trio_lengkap(r)
        if t:
            CHIP.append({"uid": r["uid"], "kelas_xbd": r["kelas_xbd"],
                         "pra": t[0], "pasca": t[1], "gabung": t[2]})

assert CHIP, "Tidak ada chip yang ketiga variannya lengkap. Periksa isi folder pra/pasca/gabung."
print("sidik fitur :", SIDIK, "| versi prompt:", VERSI)
print("chip dipakai:", len(CHIP), "dari target", N_CHIP)
for c in CHIP:
    print(f"   {c['uid'][:8]}  {c['kelas_xbd']}")

from PIL import Image
contoh = Image.open(CHIP[0]["gabung"])
print("\nukuran chip gabung :", contoh.size)
print("ukuran chip pra    :", Image.open(CHIP[0]["pra"]).size)
display(contoh)

print("\n--- 900 karakter pertama PROMPT B ---\n")
print(PROMPT["B"][:900])

## Sel 5 — muat Qwen2.5-VL-3B

Unduhan sekitar 7 GB, 2–5 menit pada percobaan pertama.

In [ ]:
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

def _muat(kelas, nama, **kw):
    # transformers baru memakai `dtype`, yang lama memakai `torch_dtype`
    try:
        return kelas.from_pretrained(nama, dtype=torch.float16, **kw)
    except TypeError:
        return kelas.from_pretrained(nama, torch_dtype=torch.float16, **kw)

qwen = _muat(Qwen2_5_VLForConditionalGeneration, MODEL_QWEN, device_map="auto").eval()
qproc = AutoProcessor.from_pretrained(MODEL_QWEN)
print("Qwen siap |", round(sum(p.numel() for p in qwen.parameters()) / 1e9, 2), "miliar parameter")

def jalan_qwen(jalur_gambar, teks):
    gambar = [Image.open(j).convert("RGB") for j in jalur_gambar]
    isi = [{"type": "image", "image": g} for g in gambar] + [{"type": "text", "text": teks}]
    pesan = [{"role": "user", "content": isi}]
    chat = qproc.apply_chat_template(pesan, tokenize=False, add_generation_prompt=True)
    masuk = qproc(text=[chat], images=gambar, return_tensors="pt", padding=True).to(qwen.device)
    with torch.inference_mode():
        keluar = qwen.generate(**masuk, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    potong = keluar[0][masuk["input_ids"].shape[1]:]
    return qproc.decode(potong, skip_special_tokens=True)

print("\n--- uji satu panggilan ---")
print(jalan_qwen([CHIP[0]["gabung"]], PROMPT["B"])[:600])

## Sel 6 — muat InternVL3-2B

Model ini memakai `trust_remote_code`, sehingga kodenya diunduh dari Hugging Face dan
lebih rentan bentrok versi dengan `transformers` bawaan Colab. Kalau sel ini gagal,
**biarkan saja dan lanjut ke sel 7** — smoke test tetap jalan dengan Qwen sendirian, dan
kegagalannya ikut tercatat. Laporkan errornya ke saya.

In [ ]:
import torch, torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

RERATA, SIMPANGAN = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def _ubah(n=448):
    return T.Compose([T.Lambda(lambda im: im.convert("RGB")),
                      T.Resize((n, n), interpolation=InterpolationMode.BICUBIC),
                      T.ToTensor(), T.Normalize(RERATA, SIMPANGAN)])

def _rasio_terdekat(ar, daftar, w, h, n):
    terbaik, beda = (1, 1), float("inf")
    for r in daftar:
        d = abs(ar - r[0] / r[1])
        if d < beda or (d == beda and w * h > 0.5 * n * n * r[0] * r[1]):
            beda, terbaik = d, r
    return terbaik

def _petak(im, n=448, maks=6, thumbnail=True):
    w, h = im.size
    daftar = sorted({(i, j) for k in range(1, maks + 1)
                     for i in range(1, k + 1) for j in range(1, k + 1)
                     if 1 <= i * j <= maks}, key=lambda x: x[0] * x[1])
    ri, rj = _rasio_terdekat(w / h, daftar, w, h, n)
    lebar, tinggi = n * ri, n * rj
    ubah = im.resize((lebar, tinggi))
    hasil = []
    for i in range(ri * rj):
        kolom, baris = i % ri, i // ri
        hasil.append(ubah.crop((kolom * n, baris * n, (kolom + 1) * n, (baris + 1) * n)))
    if thumbnail and len(hasil) != 1:
        hasil.append(im.resize((n, n)))
    return hasil

def piksel_internvl(jalur, maks):
    tf = _ubah(448)
    return torch.stack([tf(p) for p in _petak(Image.open(jalur).convert("RGB"), 448, maks)])

def _muat_ivl():
    return AutoModel.from_pretrained(MODEL_INTERNVL, torch_dtype=torch.bfloat16,
                                     low_cpu_mem_usage=True, use_flash_attn=False,
                                     trust_remote_code=True).eval().cuda()

INTERNVL_SIAP = False
try:
    try:
        ivl = _muat_ivl()
    except AttributeError as e:
        if "all_tied_weights_keys" not in str(e):
            raise
        # cadangan: sediakan atribut yang dicari pemuat transformers baru
        print("menambal all_tied_weights_keys lalu mencoba ulang ...")
        import transformers
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = property(
            lambda self: list(getattr(self, "_tied_weights_keys", None) or []))
        ivl = _muat_ivl()
    itok = AutoTokenizer.from_pretrained(MODEL_INTERNVL, trust_remote_code=True, use_fast=False)
    GEN = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    INTERNVL_SIAP = True
    print("InternVL siap")
except Exception as e:
    print(">>> InternVL GAGAL DIMUAT — lanjut saja ke sel 7, Qwen tetap jalan.")
    print(">>> ", type(e).__name__, str(e)[:400])

def jalan_internvl(jalur_gambar, teks, maks_petak):
    piksel = [piksel_internvl(j, maks_petak).to(torch.bfloat16).cuda() for j in jalur_gambar]
    jumlah = [p.size(0) for p in piksel]
    if len(jalur_gambar) == 1:
        tanya = "<image>\n" + teks
    else:
        tanya = "".join(f"Citra-{i+1}: <image>\n" for i in range(len(jalur_gambar))) + teks
    with torch.inference_mode():
        return ivl.chat(itok, torch.cat(piksel, dim=0), tanya, GEN, num_patches_list=jumlah)

if INTERNVL_SIAP:
    print("\n--- uji satu panggilan ---")
    print(jalan_internvl([CHIP[0]["gabung"]], PROMPT["B"], 6)[:600])

## Sel 7 — jalankan smoke test

Dua model × dua konfigurasi × lima chip. Perkiraan 4–8 menit di T4.
Satu panggilan yang gagal tidak menghentikan sisanya; kegagalannya ikut dicatat.

In [ ]:
import json, time, traceback

TUGAS = [("qwen2.5-vl-3b", "B", lambda c: jalan_qwen([c["gabung"]], PROMPT["B"])),
         ("qwen2.5-vl-3b", "A", lambda c: jalan_qwen([c["pra"], c["pasca"]], PROMPT["A"]))]
if INTERNVL_SIAP:
    TUGAS += [("internvl3-2b", "B", lambda c: jalan_internvl([c["gabung"]], PROMPT["B"], 6)),
              ("internvl3-2b", "A", lambda c: jalan_internvl([c["pra"], c["pasca"]], PROMPT["A"], 1))]

HASIL = []
for model, konf, fn in TUGAS:
    for c in CHIP:
        t0 = time.time()
        try:
            mentah, galat = fn(c), None
        except Exception:
            mentah, galat = None, traceback.format_exc(limit=3)
        HASIL.append({"model": model, "konfigurasi": konf, "sidik_fitur": SIDIK,
                      "uid": c["uid"], "kelas_xbd": c["kelas_xbd"],
                      "detik": round(time.time() - t0, 2),
                      "keluaran_mentah": mentah, "galat": galat})
        print(("OK " if galat is None else "!! ") +
              f"{model:<15}{konf}  {c['uid'][:8]}  {HASIL[-1]['detik']:>6.2f}s")

print("\nselesai:", len(HASIL), "panggilan")

## Sel 8 — cetak keluaran mentah

In [ ]:
import collections

for r in HASIL:
    print("=" * 78)
    print(f"model={r['model']}  konfigurasi={r['konfigurasi']}  uid={r['uid'][:8]}  "
          f"label_xBD={r['kelas_xbd']}  {r['detik']}s")
    print("-" * 78)
    print(r["galat"] if r["galat"] else r["keluaran_mentah"])
print("=" * 78)

waktu = collections.defaultdict(list)
for r in HASIL:
    if r["galat"] is None:
        waktu[(r["model"], r["konfigurasi"])].append(r["detik"])

print("\nRerata waktu per chip:")
for (m, k), v in waktu.items():
    print(f"  {m:<15}{k}  {sum(v)/len(v):>6.2f}s   (n={len(v)})")
gagal = sum(1 for r in HASIL if r["galat"])
print(f"\nPanggilan gagal: {gagal}/{len(HASIL)}")

# pemeriksaan mutu: apakah model MEMILIH, atau sekadar menyalin kerangka keluaran?
nyalin = [r for r in HASIL if r["keluaran_mentah"]
          and ("pilih satu" in r["keluaran_mentah"] or " | " in r["keluaran_mentah"])]
print(f"Keluaran yang masih menyalin kerangka (bukan memilih): {len(nyalin)}/{len(HASIL)}")
if nyalin:
    print("  >>> prompt belum berhasil; laporkan angka ini ke pembimbing kedua.")
if waktu:
    rerata = sum(sum(v) for v in waktu.values()) / sum(len(v) for v in waktu.values())
    print(f"Perkiraan run penuh 400 chip per model per konfigurasi: "
          f"{rerata*400/60:.0f} menit")

## Sel 9 — simpan ke Drive dan unduh

In [ ]:
import json
from pathlib import Path

KELUARAN = BASIS / "smoke_test_mentah.jsonl"
with open(KELUARAN, "w", encoding="utf-8") as fh:
    for r in HASIL:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print("tersimpan di Drive:", KELUARAN, f"({KELUARAN.stat().st_size/1024:.1f} KB)")

try:
    from google.colab import files
    files.download(str(KELUARAN))
except Exception as e:
    print("unduhan otomatis gagal, ambil manual dari Drive:", e)